# 03 — Silver SIAF: tabla `siaf_ingreso`

Construye la tabla Silver del SIAF (presupuesto y ejecución de ingresos) a partir de los 18 CSV de Bronze, con lo visto en `02_bronze_perfilado`.

| Parte | Qué hace |
| --- | --- |
| 0 | Configuración |
| 1 | Archivos y columnas: qué se usa y qué no |
| 2 | Carga cruda |
| 3 | Diagnóstico inicial (antes de limpiar) |
| 4 | Limpieza paso a paso |
| 5 | Reglas de calidad (todas en una celda, con un resumen); 5.1 cuadre |
| 6 | Escritura en Parquet (Silver y cuarentena) |
| 7 | Bitácora (formato de auditoría) y `data/auditoria/volumenes.parquet` |
| 8 | Resumen final |

**Grano:** una fila por año, mes, ejecutora (`sec_ejec`) y clasificador de ingreso (fuente, rubro, tipo de recurso, genérica, subgenérica, específica y sus detalles). Se conserva el grano **mensual** del SIAF; los totales anuales se arman en Gold.

## 0. Configuración

In [1]:
import sys
from pathlib import Path
RAIZ = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "ARQUITECTURA.md").exists())
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

import re, uuid
from datetime import datetime, timezone
from functools import reduce
import pandas as pd
from pyspark import StorageLevel
from pyspark.sql import functions as F

from src.config import cargar_config, cargar_reglas, ruta
from src.spark import crear_sesion
from src import bronze, auditoria
from src.perfilado import detectar_formato, leer_csv, a_pandas
from src.silver import aplicar

pd.set_option("display.max_columns", 40); pd.set_option("display.width", 250); pd.set_option("display.max_colwidth", 70)
cfg = cargar_config()
REGLAS = {r["id"]: r for r in cargar_reglas()}
spark = crear_sesion("silver_siaf")
spark.sparkContext.setLogLevel("ERROR")
CAMBIOS = []
inicio = datetime.now(timezone.utc)
RUN_ID = f"silver_siaf_{inicio:%Y%m%dT%H%M%SZ}_{uuid.uuid4().hex[:6]}"
TABLA = "siaf_ingreso"
ANIOS_PARCIALES = cfg["anios"]["parcial"]

man = pd.DataFrame(bronze.leer_manifest()["archivos"])
f1 = man[(man["fuente"] == "F1") & (man["origen"] == "descarga")].copy()
f1["anio"] = f1["anio"].astype(int)
f1 = f1.sort_values("anio").reset_index(drop=True)

def contar(cond):
    return F.sum(F.coalesce(cond, F.lit(False)).cast("long"))

print("run_id:", RUN_ID, "| archivos:", len(f1))

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/22 15:36:45 WARN Utils: Your hostname, Rodrigo, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/11/22 15:36:45 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/11/22 15:36:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/22 15:36:46 WARN SparkConf: Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in standalone/kubernetes and LOCAL_DIRS in YARN).


run_id: silver_siaf_20261122T203648Z_543d24 | archivos: 18


## 1. Archivos y columnas: qué se usa y qué no

In [2]:
f1_tabla = f1.assign(MB=(f1["bytes"] / 1e6).round(1), filas=f1["filas"].astype(int),
                     uso=f1["archivo"].map(lambda a: "Sí (archivo Diario, D-005)" if "Diario" in a else "Sí"))
display(f1_tabla[["anio", "archivo", "filas", "MB", "fecha_descarga", "uso"]].set_index("anio"))
cols_por_archivo = {r.archivo: detectar_formato(RAIZ / r.ruta)["columnas"] for r in f1.itertuples()}
COLS = next(iter(cols_por_archivo.values()))
print(f"Los {len(f1)} archivos tienen las mismas {len(COLS)} columnas en el mismo orden:",
      all(c == COLS for c in cols_por_archivo.values()))

,archivo,filas,MB,fecha_descarga,uso
anio,,,,,
2009,2009-Ingreso.csv,1235428,578.9,2026-11-20,Sí
2010,2010-Ingreso.csv,1320104,622.4,2026-11-20,Sí
2011,2011-Ingreso.csv,1348362,637.2,2026-11-20,Sí
2012,2012-Ingreso.csv,759578,367.0,2026-11-20,Sí
2013,2013-Ingreso.csv,721663,348.6,2026-11-20,Sí
2014,2014-Ingreso.csv,753998,364.8,2026-11-20,Sí
2015,2015-Ingreso.csv,781007,380.7,2026-11-20,Sí
2016,2016-Ingreso.csv,764731,372.3,2026-11-20,Sí
2017,2017-Ingreso.csv,683816,332.7,2026-11-20,Sí


Los 18 archivos tienen las mismas 36 columnas en el mismo orden: True


**Archivos:** se usan **los 18**. 2009–2024 son los archivos anuales y 2025–2026 son los *Diario*. Los *Mensual* de 2025 y 2026 no se descargan: en 2025 el Mensual repite al Diario y en 2026 está desactualizado (D-005).

**Columnas** (las 36 son iguales en los 18 archivos):

| Grupo | Columnas | ¿Se conserva? |
| --- | --- | --- |
| Período | `ANO_DOC`, `MES_DOC` | Sí |
| Nivel de gobierno | `NIVEL_GOBIERNO` y su nombre | Se usa para filtrar y luego se quita (paso 4.2) |
| Sector y pliego | `SECTOR`, `PLIEGO` y sus nombres | No: vacías en todo el nivel M (paso 4.2) |
| Ejecutora | `SEC_EJEC`, `EJECUTORA`, `EJECUTORA_NOMBRE` | Sí: `SEC_EJEC` conecta con el SISMEPRE |
| Ubicación | departamento, provincia, distrito (código y nombre) | Sí: de aquí sale el `ubigeo` (conexión con el RENAMU) |
| Clasificador de ingreso | fuente, rubro, tipo de recurso, genérica, subgenérica, específica y sus detalles (código y nombre) | Sí |
| Montos | `MONTO_PIA`, `MONTO_PIM`, `MONTO_RECAUDADO` | Sí |

**Filas:** el SIAF trae **todo el Estado**. Solo se usa el nivel **M (Gobierno Local)** y, dentro de él, las **municipalidades**:
- **E (nacional) y R (regional):** no son municipalidades, no tienen un `ubigeo` de distrito propio y no están en el RENAMU ni en el SISMEPRE.
- **Mancomunidades:** son asociaciones de varias municipalidades. No son un distrito y usan el `ubigeo` de su sede, que choca con el de la municipalidad. Además, buena parte de sus ingresos son transferencias de sus socias, así que sumarlas contaría dos veces el mismo dinero.

Lo que se excluye **no se pierde en silencio**: se cuenta como *fuera de alcance* y entra en el cuadre (5.1).

## 2. Carga cruda

Los 18 archivos se leen juntos, todo como texto. Cada fila guarda su archivo de origen, el `sha256` y la fecha de descarga del manifest.

In [3]:
FMT = detectar_formato(RAIZ / f1.iloc[0]["ruta"])
origen = spark.createDataFrame(f1[["ruta", "anio", "sha256", "fecha_descarga"]].rename(columns={
    "ruta": "_archivo_origen", "anio": "anio_archivo", "sha256": "_sha256_origen", "fecha_descarga": "_fecha_descarga"}))
crudo = (leer_csv(spark, [str(RAIZ / r) for r in f1["ruta"]], FMT)
         .withColumn("_archivo_origen", F.regexp_extract(F.input_file_name(), r"(data/bronze/.*)$", 1))
         .join(F.broadcast(origen), "_archivo_origen"))

ES_M = F.col("NIVEL_GOBIERNO") == "M"
ES_MUNI = ES_M & F.col("EJECUTORA_NOMBRE").startswith("MUNICIPALIDAD")
num = lambda c: F.trim(F.col(c)).cast("decimal(18,2)")
MONTOS_RAW = ["MONTO_PIA", "MONTO_PIM", "MONTO_RECAUDADO"]

# Una sola pasada sobre los 7 GB: conteos por año para la carga y el diagnóstico (parte 3)
agg = a_pandas(crudo.groupBy("anio_archivo").agg(
    F.count("*").alias("filas"),
    contar(F.col("NIVEL_GOBIERNO") == "E").alias("E_nacional"), contar(F.col("NIVEL_GOBIERNO") == "R").alias("R_regional"),
    contar(ES_MUNI).alias("M_municipalidades"), contar(ES_M & F.col("EJECUTORA_NOMBRE").startswith("MANCOMUNIDAD")).alias("M_mancomunidades"),
    contar(ES_M & ~F.col("EJECUTORA_NOMBRE").startswith("MUNICIPALIDAD") & ~F.col("EJECUTORA_NOMBRE").startswith("MANCOMUNIDAD")).alias("M_otras"),
    contar(ES_MUNI & reduce(lambda a, b: a | b, [F.length(F.trim(c)) == 1 for c in ("DEPARTAMENTO_EJECUTORA", "PROVINCIA_EJECUTORA", "DISTRITO_EJECUTORA")])).alias("codigo_de_1_digito"),
    contar(ES_MUNI & reduce(lambda a, b: a | b, [F.trim(c) == "" for c in ("DEPARTAMENTO_EJECUTORA", "PROVINCIA_EJECUTORA", "DISTRITO_EJECUTORA")])).alias("ubicacion_vacia"),
    contar(ES_MUNI & (F.col("ANO_DOC").cast("int") != F.col("anio_archivo"))).alias("ano_doc_distinto"),
    contar(ES_MUNI & ~F.col("MES_DOC").cast("int").between(1, 12)).alias("mes_fuera_de_rango"),
    contar(ES_MUNI & reduce(lambda a, b: a | b, [num(c).isNull() & (F.trim(c) != "") for c in MONTOS_RAW])).alias("monto_no_numerico"),
    contar(ES_MUNI & (num("MONTO_RECAUDADO") < 0)).alias("recaudado_negativo"),
    contar(ES_MUNI & reduce(lambda a, b: a & b, [num(c) == 0 for c in MONTOS_RAW])).alias("montos_todos_en_cero"),
    *[F.sum(F.when(ES_MUNI, num(c))).alias(f"bronze_{c.lower()}") for c in MONTOS_RAW],
).orderBy("anio_archivo")).set_index("anio_archivo")
agg.index.name = "anio"
carga = agg[["filas"]].join(f1.set_index("anio")["filas"].astype(int).rename("filas_manifest"))
carga["iguales"] = carga["filas"] == carga["filas_manifest"]
print(f"Filas cargadas: {int(carga['filas'].sum()):,} de {len(f1)} archivos · conteo de Spark = manifest en todos:", carga["iguales"].all())

Filas cargadas: 15,204,055 de 18 archivos · conteo de Spark = manifest en todos: True


## 3. Diagnóstico inicial (antes de limpiar)

### 3.1 Alcance: niveles de gobierno y tipos de ejecutora

In [4]:
alc = agg[["filas", "E_nacional", "R_regional", "M_municipalidades", "M_mancomunidades", "M_otras"]].copy()
alc["pct_municipalidades"] = (100 * alc["M_municipalidades"] / alc["filas"]).round(1)
display(alc.drop(columns="M_otras") if (alc["M_otras"] == 0).all() else alc)

,filas,E_nacional,R_regional,M_municipalidades,M_mancomunidades,pct_municipalidades
anio,,,,,,
2009,1235428,64551,90407,1080470,0,87.5
2010,1320104,65945,94370,1159789,0,87.9
2011,1348362,66046,97197,1185119,0,87.9
2012,759578,50067,70586,638761,164,84.1
2013,721663,54362,76959,590028,314,81.8
2014,753998,59067,83258,611366,307,81.1
2015,781007,61445,86962,632282,318,81.0
2016,764731,62103,89084,613187,357,80.2
2017,683816,54945,77922,550641,308,80.5


**Diagnóstico:** cerca del 84 % de las filas son de municipalidades. Las mancomunidades aparecen desde 2012. Todo lo que no es municipalidad del nivel M va a *fuera de alcance* (paso 4.1).

### 3.2 Identificación y período (solo municipalidades)

In [5]:
ident = agg[["codigo_de_1_digito", "ubicacion_vacia", "ano_doc_distinto", "mes_fuera_de_rango"]]
sin = [c for c in ident.columns if (ident[c] == 0).all()]
display(ident.drop(columns=sin)) if len(sin) < len(ident.columns) else None
print("Sin casos en ningún año:", ", ".join(sin) or "—")

Sin casos en ningún año: codigo_de_1_digito, ubicacion_vacia, ano_doc_distinto, mes_fuera_de_rango


**Diagnóstico:** en las municipalidades la ubicación viene completa, con códigos de 2 dígitos, y el año y el mes son válidos. Los códigos de 1 dígito que mostró el perfilado (2012) eran de mancomunidades, que quedan fuera de alcance.

In [6]:
# Base de trabajo: solo municipalidades del nivel M (se guarda en disco para no releer los 7 GB en cada paso)
LLAVE_RAW = ["ANO_DOC", "MES_DOC", "SEC_EJEC", "FUENTE_FINANCIAMIENTO", "RUBRO", "TIPO_RECURSO",
             "GENERICA", "SUBGENERICA", "SUBGENERICA_DET", "ESPECIFICA", "ESPECIFICA_DET"]
muni = crudo.where(ES_MUNI).persist(StorageLevel.DISK_ONLY)
print(f"Municipalidades del nivel M: {muni.count():,} filas")

Municipalidades del nivel M: 12,746,440 filas


### 3.3 Ejecutoras que cambian de nombre dentro del año

Cada caso con los meses en que se usó cada nombre. En Silver se conservan tal como vienen; Gold usa el último.

In [7]:
nom = a_pandas(muni.groupBy("anio_archivo", "SEC_EJEC", "EJECUTORA_NOMBRE")
                  .agg(F.min(F.col("MES_DOC").cast("int")).alias("desde_mes"), F.max(F.col("MES_DOC").cast("int")).alias("hasta_mes")))
varios = nom.groupby(["anio_archivo", "SEC_EJEC"])["EJECUTORA_NOMBRE"].transform("nunique") > 1
cambios_nombre = nom[varios].sort_values(["anio_archivo", "SEC_EJEC", "desde_mes"]).rename(columns={"anio_archivo": "anio"})
print(f"{cambios_nombre[['anio', 'SEC_EJEC']].drop_duplicates().shape[0]} casos (año × ejecutora) con más de un nombre")
display(cambios_nombre.set_index(["anio", "SEC_EJEC"]))

11 casos (año × ejecutora) con más de un nombre


EJECUTORA_NOMBRE  desde_mes  hasta_mes
anio SEC_EJEC                                                                            
2013 301266                   MUNICIPALIDAD DISTRITAL DE LOS OLIVOS          1         12
     301266    MUNICIPALIDAD DISTRITAL DE LOS OLIVOS (LAS PALMERAS)          1          1
2014 300749                     MUNICIPALIDAD DISTRITAL DE ECHARATI          1         12
     300749                     MUNICIPALIDAD DISTRITAL DE ECHARATE          9          9
     300772                    MUNICIPALIDAD DISTRITAL DE KOSÑIPATA          1         12
     300772                   MUNICIPALIDAD DISTRITAL DE KCOSÑIPATA          1          8
     300985                       MUNICIPALIDAD PROVINCIAL DE NAZCA          1         12
     300985                       MUNICIPALIDAD PROVINCIAL DE NASCA          1          1
2016 301250                     MUNICIPALIDAD METROPOLITANA DE LIMA          1         12
     301250                        MUNICIPALIDAD PROVINCIAL DE LIMA          7          7
2017 301266                   MUNICIPALIDAD DISTRITAL DE LOS OLIVOS          1         12
     301266    MUNICIPALIDAD DISTRITAL DE LOS OLIVOS (LAS PALMERAS)         10         10
2018 301843            MUNICIPALIDAD DISTRITAL CANOAS DE PUNTA SAL           1         12
     301843             MUNICIPALIDAD DISTRITAL CANOAS DE PUNTA SAL         12         12
2020 300939                       MUNICIPALIDAD DISTRITAL DE MOLINO          1         12
     300939                      MUNICIPALIDAD DISTRITAL DE MOLINOS          1          1
2021 300939                       MUNICIPALIDAD DISTRITAL DE MOLINO          1         12
     300939                      MUNICIPALIDAD DISTRITAL DE MOLINOS          5          5
2023 301568                   MUNICIPALIDAD DISTRITAL DE  EL ARENAL          1         12
     301568                       MUNICIPALIDAD DISTRITAL DE ARENAL          3          3
2025 300939                       MUNICIPALIDAD DISTRITAL DE MOLINO          1         12
     300939                      MUNICIPALIDAD DISTRITAL DE MOLINOS          7         10

**Diagnóstico:** casi todos son correcciones de texto: NAZCA → NASCA, ECHARATI → ECHARATE, MOLINO → MOLINOS, un espacio doble en EL ARENAL, y Lima, que en julio de 2016 figuró como PROVINCIAL. En Silver cada fila conserva el nombre que trae el SIAF; Gold toma el último de cada ejecutora.

### 3.4 Filas repetidas por la llave de negocio

Llave: año, mes, ejecutora, fuente, rubro, tipo de recurso y clasificador completo. Se revisa cuántas hay y en qué columnas difieren las copias.

In [8]:
from pyspark.sql import Window
rep = (muni.withColumn("_n", F.count("*").over(Window.partitionBy(*LLAVE_RAW))).where("_n > 1")
           .withColumn("_llave", F.concat_ws("|", *LLAVE_RAW)).persist(StorageLevel.DISK_ONLY))
por_anio = a_pandas(rep.groupBy("anio_archivo").agg(F.count("*").alias("filas_repetidas"), F.countDistinct("_llave").alias("llaves"))
                       .orderBy("anio_archivo")).set_index("anio_archivo")
display(por_anio.T)
OTRAS = [c for c in COLS if c not in LLAVE_RAW + MONTOS_RAW]
difieren = a_pandas(rep.groupBy("_llave").agg(*[(F.countDistinct(c) > 1).cast("int").alias(c) for c in OTRAS + MONTOS_RAW])
                       .agg(*[F.sum(c).alias(c) for c in OTRAS + MONTOS_RAW])).T[0]
print("Columnas que cambian entre las copias (n° de llaves):", difieren[difieren > 0].to_dict() or "ninguna")
ej = a_pandas(rep.where(F.col("_llave") == rep.select("_llave").first()[0])
                 .select("_llave", *[F.col(c) for c in ["EJECUTORA_NOMBRE", "RUBRO_NOMBRE", "ESPECIFICA_DET_NOMBRE"] + MONTOS_RAW]))
print("Ejemplo: las copias de una llave")
display(ej.drop(columns="_llave"))

anio_archivo,2013,2014,2016,2017,2018,2020,2021,2023,2025
filas_repetidas,2,42,10,2,6,2,2,4,4
llaves,1,21,5,1,3,1,1,2,2


Columnas que cambian entre las copias (n° de llaves): {'EJECUTORA_NOMBRE': 36, 'TIPO_RECURSO_NOMBRE': 1, 'MONTO_PIA': 2, 'MONTO_PIM': 3, 'MONTO_RECAUDADO': 35}
Ejemplo: las copias de una llave


,EJECUTORA_NOMBRE,RUBRO_NOMBRE,ESPECIFICA_DET_NOMBRE,MONTO_PIA,MONTO_PIM,MONTO_RECAUDADO
0,MUNICIPALIDAD PROVINCIAL DE NAZCA,RECURSOS DIRECTAMENTE RECAUDADOS,"AUTORIZACION, INSPECCION Y CONTROL SANITARIO",0,0,1061.95
1,MUNICIPALIDAD PROVINCIAL DE NASCA,RECURSOS DIRECTAMENTE RECAUDADOS,"AUTORIZACION, INSPECCION Y CONTROL SANITARIO",0,0,0


**Diagnóstico:** en las municipalidades son **74 filas en 37 llaves** (2 copias por llave); las 291 del perfilado incluían todos los niveles de gobierno. Las copias aparecen **cuando la ejecutora cambió de nombre a mitad de año** (3.3): la misma partida quedó registrada una vez con el nombre anterior y otra con el nuevo. Por eso difieren en el nombre (36 de las 37 llaves) y en los montos. Se consolidan **sumando los montos**, y la fila queda con el **nombre principal del año** de esa ejecutora, el que se usó en más meses (paso 4.7).

### 3.5 Montos (solo municipalidades)

In [9]:
mont = agg[["monto_no_numerico", "recaudado_negativo", "montos_todos_en_cero"]].copy()
mont["pct_todo_en_cero"] = (100 * mont["montos_todos_en_cero"] / agg["M_municipalidades"]).round(1)
display(mont.drop(columns=[c for c in ["monto_no_numerico"] if (mont[c] == 0).all()]))
print("Montos no numéricos:", int(mont["monto_no_numerico"].sum()))

,recaudado_negativo,montos_todos_en_cero,pct_todo_en_cero
anio,,,
2009,1575,316528,29.3
2010,2264,378458,32.6
2011,2159,383892,32.4
2012,2343,126977,19.9
2013,2295,117586,19.9
2014,2830,120431,19.7
2015,2556,118533,18.7
2016,3144,116221,19.0
2017,3521,35629,6.5


Montos no numéricos: 0


**Diagnóstico:** todos los montos son numéricos. Los recaudados negativos son devoluciones y extornos: se conservan y se registran (regla `SLV-F1-007`). Las filas con todos los montos en 0 **se conservan** (decisión del usuario): no cambian ninguna suma y muestran que la partida existía.

## 4. Limpieza paso a paso

### 4.1 Alcance

Se quedan las municipalidades del nivel M. El resto (E, R y mancomunidades) se cuenta como *fuera de alcance* por año.

In [10]:
fuera = agg[["filas", "M_municipalidades"]].assign(fuera_de_alcance=lambda d: d["filas"] - d["M_municipalidades"])
print(f"Quedan {int(fuera['M_municipalidades'].sum()):,} filas · fuera de alcance: {int(fuera['fuera_de_alcance'].sum()):,}")

Quedan 12,746,440 filas · fuera de alcance: 2,457,615


### 4.2 Columnas

**Decisión:** se quitan 6 columnas que en el nivel M no tienen información. `NIVEL_GOBIERNO` y su nombre valen siempre "M / Gobierno Local" después del filtro. `SECTOR`, `PLIEGO` y sus nombres vienen vacíos en el 100 % de las filas del nivel M (perfilado 2.3). No se pierde ningún dato. El resto pasa a minúsculas; `ANO_DOC` y `MES_DOC` se llaman `anio` y `mes`.

In [11]:
QUITAR = ["NIVEL_GOBIERNO", "NIVEL_GOBIERNO_NOMBRE", "SECTOR", "SECTOR_NOMBRE", "PLIEGO", "PLIEGO_NOMBRE"]
vacias = a_pandas(muni.agg(*[contar(F.trim(c) != "").alias(c) for c in QUITAR[2:]])).iloc[0]
print("Valores no vacíos en SECTOR/PLIEGO dentro del nivel M:", vacias.to_dict())
RENOMBRE = {"ANO_DOC": "anio", "MES_DOC": "mes"}
t = muni.select(*[F.col(c).alias(RENOMBRE.get(c, c.lower())) for c in COLS if c not in QUITAR],
                "anio_archivo", "_archivo_origen", "_sha256_origen", "_fecha_descarga")
TEXTO = [c for c in t.columns if not c.startswith("monto_") and c not in ("anio_archivo",) and not c.startswith("_")]
print(f"Columnas: {len(COLS)} → {len(COLS) - len(QUITAR)}")

Valores no vacíos en SECTOR/PLIEGO dentro del nivel M: {'SECTOR': 0, 'SECTOR_NOMBRE': 0, 'PLIEGO': 0, 'PLIEGO_NOMBRE': 0}
Columnas: 36 → 30


### 4.3 Espacios sobrantes

Se quitan los espacios al inicio y al final y los dobles dentro del texto.

In [12]:
t = aplicar(t, "4.3 espacios", {c: F.trim(F.regexp_replace(F.col(c), r"\s+", " ")) for c in TEXTO}, CAMBIOS,
           por="anio_archivo", checkpoint=False)

4.3 espacios: valores cambiados por anio_archivo y columna


,ejecutora_nombre,distrito_ejecutora_nombre,tipo_recurso_nombre,subgenerica_det_nombre,especifica_nombre,especifica_det_nombre
anio_archivo,,,,,,
2009,1777,545,51,20068,1173,26764
2010,2160,535,19,18381,1405,27182
2011,2339,523,108,16029,1237,26037
2012,954,188,7,6095,456,22576
2013,833,176,964,5275,419,13495
2014,1587,142,4234,5194,319,12275
2015,1432,120,4948,5350,360,10993
2016,1358,167,4572,5395,556,10778
2017,1153,150,2907,3670,390,7204


Ejemplos (antes → después):


,anio_archivo,columna,antes,después
0,2009,ejecutora_nombre,«MUNICIPALIDAD DISTRITAL DE SAN JOSE DE LOS CHORRILLOS - CUENCA»,«MUNICIPALIDAD DISTRITAL DE SAN JOSE DE LOS CHORRILLOS - CUENCA»
1,2009,distrito_ejecutora_nombre,«PACHAMARCA »,«PACHAMARCA»


### 4.4 `ubigeo`

`ubigeo` = departamento + provincia + distrito, cada uno completado a 2 dígitos por seguridad (`1` → `01`). Es la llave de conexión con el RENAMU.

In [13]:
UB = ["departamento_ejecutora", "provincia_ejecutora", "distrito_ejecutora"]
t = t.withColumn("ubigeo", F.concat(*[F.lpad(c, 2, "0") for c in UB]))
x = a_pandas(t.where(reduce(lambda a, b: a | b, [F.length(c) == 1 for c in UB]))
               .groupBy("anio_archivo").agg(F.count("*").alias("filas_completadas"), F.countDistinct("sec_ejec").alias("ejecutoras")))
if len(x):
    display(x.set_index("anio_archivo")); display(a_pandas(t.where(F.length("departamento_ejecutora") == 1).select(*UB, "ubigeo").limit(1)))
else:
    print("Ningún código con 1 dígito: no hay que completar ceros")
display(a_pandas(t.select(*UB, "ubigeo").limit(1)))

Ningún código con 1 dígito: no hay que completar ceros


,departamento_ejecutora,provincia_ejecutora,distrito_ejecutora,ubigeo
0,01,01,01,010101


### 4.5 Tipo de municipalidad

Sale del nombre de la ejecutora: *MUNICIPALIDAD PROVINCIAL* o *DISTRITAL*. Lima figura como *METROPOLITANA* en la mayoría de los años y como *PROVINCIAL DE LIMA* en 2010–2012; se etiqueta siempre **PROVINCIAL**, igual que el tipo 1 del RENAMU, para que su tipo no cambie de un año a otro.

In [14]:
t = t.withColumn("tipo_municipalidad", F.when(F.col("ejecutora_nombre").startswith("MUNICIPALIDAD PROVINCIAL"), "PROVINCIAL")
                                        .when(F.col("ejecutora_nombre").startswith("MUNICIPALIDAD DISTRITAL"), "DISTRITAL")
                                        .when(F.col("ejecutora_nombre").startswith("MUNICIPALIDAD METROPOLITANA"), "PROVINCIAL"))
display(a_pandas(t.groupBy("anio_archivo").pivot("tipo_municipalidad").agg(F.countDistinct("sec_ejec")).orderBy("anio_archivo"))
        .set_index("anio_archivo").fillna(0).astype(int).T)

anio_archivo,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026
DISTRITAL,1639,1639,1642,1643,1644,1651,1658,1677,1678,1678,1678,1678,1694,1694,1695,1695,1696,1696
PROVINCIAL,195,195,195,195,195,196,196,196,196,196,196,196,196,196,196,196,196,196


### 4.6 Tipos de dato

`anio` y `mes` pasan a entero y los montos a `decimal(18,2)`. Si un valor no se puede convertir queda nulo y se marca para la regla `SLV-F1-001`.

In [15]:
MONTOS = ["monto_pia", "monto_pim", "monto_recaudado"]
t = (t.withColumn("_no_convertidos", reduce(lambda a, b: a + b, [(F.col(c).isNotNull() & (F.col(c) != "") & F.col(c).cast("decimal(18,2)").isNull()).cast("int") for c in MONTOS]))
      .withColumn("anio", F.col("anio").cast("int")).withColumn("mes", F.col("mes").cast("int")))
t = t.select(*[F.col(c).cast("decimal(18,2)").alias(c) if c in MONTOS else F.col(c) for c in t.columns])
print("Valores no convertidos:", int(a_pandas(t.agg(F.sum("_no_convertidos").alias("n")))["n"][0]))
t.select("anio", "mes", *MONTOS).printSchema()

Valores no convertidos: 0
root
 |-- anio: integer (nullable = true)
 |-- mes: integer (nullable = true)
 |-- monto_pia: decimal(18,2) (nullable = true)
 |-- monto_pim: decimal(18,2) (nullable = true)
 |-- monto_recaudado: decimal(18,2) (nullable = true)



### 4.7 Consolidación de filas repetidas

Las copias de una misma llave (3.4) se juntan en una fila **sumando los montos**. Si las copias traen nombres distintos, la fila queda con el nombre principal del año. `_filas_origen` dice cuántas filas de Bronze forman cada fila. Las filas que fallarían una regla crítica (montos no convertibles, año o mes inválido, `ubigeo` inválido) no se consolidan: pasan tal cual para que la regla las mande a cuarentena.

In [16]:
LLAVE = ["anio", "mes", "sec_ejec", "fuente_financiamiento", "rubro", "tipo_recurso",
         "generica", "subgenerica", "subgenerica_det", "especifica", "especifica_det"]
critica = ((F.col("_no_convertidos") > 0) | (F.col("anio") != F.col("anio_archivo")) | ~F.col("mes").between(1, 12)
           | ~F.coalesce(F.col("ubigeo").rlike(r"^\d{6}$"), F.lit(False)))
DESCR = [c for c in t.columns if c not in LLAVE + MONTOS + ["_no_convertidos", "ejecutora_nombre"]]
# nombre principal del año: el que la ejecutora usó en más meses (solo se usa si las copias traen nombres distintos)
w_nom = Window.partitionBy("anio", "sec_ejec").orderBy(F.desc("meses"), F.desc("ejecutora_nombre"))
principal = (t.groupBy("anio", "sec_ejec", "ejecutora_nombre").agg(F.countDistinct("mes").alias("meses"))
              .withColumn("_r", F.row_number().over(w_nom)).where("_r = 1").select("anio", "sec_ejec", F.col("ejecutora_nombre").alias("_nombre_principal")))
ok = (t.where(~critica).groupBy(*LLAVE)
        .agg(*[F.max(c).alias(c) for c in DESCR], F.max("ejecutora_nombre").alias("ejecutora_nombre"),
             F.countDistinct("ejecutora_nombre").alias("_n_nombres"),
             *[F.sum(c).cast("decimal(18,2)").alias(c) for c in MONTOS],
             F.count("*").alias("_filas_origen"), F.lit(0).alias("_no_convertidos"))
        .join(F.broadcast(principal), ["anio", "sec_ejec"], "left")
        .withColumn("ejecutora_nombre", F.when(F.col("_n_nombres") > 1, F.col("_nombre_principal")).otherwise(F.col("ejecutora_nombre")))
        .drop("_n_nombres", "_nombre_principal"))
malas = t.where(critica).withColumn("_filas_origen", F.lit(1))
tabla = ok.unionByName(malas).persist(StorageLevel.DISK_ONLY)
cons = a_pandas(tabla.groupBy("anio").agg(F.sum("_filas_origen").alias("filas_antes"), F.count("*").alias("filas_despues"),
                                          contar(F.col("_filas_origen") > 1).alias("filas_consolidadas")).orderBy("anio")).set_index("anio")
display(cons[cons["filas_consolidadas"] > 0].T)
display(a_pandas(tabla.where("_filas_origen > 1").select("anio", "mes", "sec_ejec", "ejecutora_nombre", "rubro", "especifica_det", "_filas_origen", *MONTOS).limit(3)))
print(f"Total: {int(cons['filas_antes'].sum()):,} → {int(cons['filas_despues'].sum()):,} filas")

anio,2013,2014,2016,2017,2018,2020,2021,2023,2025
filas_antes,590028,611366,613187,550641,527513,572281,691480,630558,739690
filas_despues,590027,611345,613182,550640,527510,572280,691479,630556,739688
filas_consolidadas,1,21,5,1,3,1,1,2,2


,anio,mes,sec_ejec,ejecutora_nombre,rubro,especifica_det,_filas_origen,monto_pia,monto_pim,monto_recaudado
0,2014,1,300985,MUNICIPALIDAD PROVINCIAL DE NAZCA,09,99,2,0.00,0.00,13872.00
1,2014,1,300772,MUNICIPALIDAD DISTRITAL DE KOSÑIPATA,09,1,2,0.00,0.00,0.00
2,2014,8,300772,MUNICIPALIDAD DISTRITAL DE KOSÑIPATA,09,18,2,0.00,0.00,7631.30


Total: 12,746,440 → 12,746,403 filas


### 4.8 Año parcial

`anio_parcial` = verdadero para los años en curso (`config.yaml → anios.parcial`). Así Gold no compara un año incompleto con uno cerrado.

In [17]:
tabla = tabla.withColumn("anio_parcial", F.col("anio").isin(ANIOS_PARCIALES))
display(a_pandas(tabla.where("anio_parcial").groupBy("anio").agg(F.max("mes").alias("ultimo_mes"))))

,anio,ultimo_mes
0,2026,11


## 5. Reglas de calidad

Las reglas están en `config.yaml`. Todas se ejecutan en la celda siguiente, que muestra un solo resumen. Las de tipo *detener* paran el notebook después de mostrarlo.

| # | Regla | Qué verifica | Acción si falla |
| --- | --- | --- | --- |
| 1 | `SLV-F1-001` | Montos convertibles a decimal | Cuarentena |
| 2 | `SLV-F1-002` | Año = año del archivo y mes entre 1 y 12 | Cuarentena |
| 3 | `SLV-F1-003` | `ubigeo` de 6 dígitos | Cuarentena |
| 4 | `SLV-F1-004` | Alcance: lo que no es municipalidad del nivel M se cuenta | Registrar |
| 5 | `SLV-F1-005` | Llave de negocio única (las copias se consolidaron) | Registrar |
| 6 | `SLV-F1-006` | Un `sec_ejec` tiene un solo `ubigeo` por año | Registrar |
| 7 | `SLV-F1-007` | Recaudado negativo (devoluciones) | Registrar |
| 8 | `SLV-TODAS-001` | Privacidad: la salida no tiene datos personales | Detener |
| 9 | `SLV-F1-008` · `SLV-TODAS-002` | Cuadre de montos y de filas (detalle en 5.1) | Detener |

- **Cuarentena:** la fila no entra a Silver; se guarda con `_regla` y `_motivo`.
- **Registrar:** la fila entra a Silver y el id de la regla queda en `_alertas`.

In [18]:
VERIF = []
def resultado(id_, evaluadas, fallidas, nota=""):
    r = REGLAS[id_]
    estado = "OK" if fallidas == 0 else {"cuarentena": "CUARENTENA", "registrar": "ALERTA", "detener": "DETENIDO"}[r["accion"]]
    VERIF.append({"regla": id_, "descripcion": r["descripcion"], "severidad": r["severidad"], "accion": r["accion"],
                  "filas_evaluadas": int(evaluadas), "filas_fallidas": int(fallidas), "estado": estado, "nota": nota})

falla = {"SLV-F1-001": F.col("_no_convertidos") > 0,
         "SLV-F1-002": (F.col("anio") != F.col("anio_archivo")) | ~F.coalesce(F.col("mes").between(1, 12), F.lit(False)),
         "SLV-F1-003": ~F.coalesce(F.col("ubigeo").rlike(r"^\d{6}$"), F.lit(False))}
motivo = {"SLV-F1-001": "monto no convertible", "SLV-F1-002": "año o mes inválido", "SLV-F1-003": "ubigeo inválido"}
regla_col = reduce(lambda acc, kv: acc.when(kv[1], F.lit(kv[0])), falla.items(), F.when(F.lit(False), F.lit(None)))
w_ub = Window.partitionBy("anio", "sec_ejec")
tabla = (tabla.withColumn("_regla", regla_col)
              .withColumn("_motivo", reduce(lambda acc, kv: acc.when(F.col("_regla") == kv[0], F.lit(kv[1])), motivo.items(), F.when(F.lit(False), F.lit(None))))
              .withColumn("_destino", F.when(F.col("_regla").isNotNull(), "cuarentena").otherwise("silver"))
              .withColumn("_f005", F.col("_filas_origen") > 1)
              .withColumn("_f006", F.size(F.collect_set("ubigeo").over(w_ub)) > 1)
              .withColumn("_f007", F.col("monto_recaudado") < 0))
tabla = tabla.withColumn("_alertas", F.concat_ws(",", *[F.when(F.col(c) & (F.col("_destino") == "silver"), F.lit(r))
                                                       for c, r in (("_f005", "SLV-F1-005"), ("_f006", "SLV-F1-006"), ("_f007", "SLV-F1-007"))]))
tabla = tabla.withColumn("_alertas", F.when(F.col("_alertas") != "", F.col("_alertas")))

n = a_pandas(tabla.agg(F.count("*").alias("filas"), F.sum("_filas_origen").alias("filas_origen"),
                       *[contar(c).alias(k) for k, c in falla.items()],
                       contar(F.col("_f005")).alias("f005"), F.sum(F.when(F.col("_f005"), F.col("_filas_origen"))).alias("f005_origen"),
                       F.countDistinct(F.when(F.col("_f006"), F.concat_ws("-", "anio", "sec_ejec"))).alias("f006"),
                       contar(F.col("_f007")).alias("f007"),
                       contar(F.col("_destino") == "silver").alias("silver"), contar(F.col("_destino") == "cuarentena").alias("cuarentena"))).iloc[0]
for k in falla:
    resultado(k, n["filas_origen"], n[k])
resultado("SLV-F1-004", int(agg["filas"].sum()), int(fuera["fuera_de_alcance"].sum()),
          f"E: {int(agg['E_nacional'].sum()):,} · R: {int(agg['R_regional'].sum()):,} · mancomunidades: {int(agg['M_mancomunidades'].sum()):,}")
resultado("SLV-F1-005", n["filas_origen"], n["f005_origen"] or 0, f"{n['f005']} filas consolidadas (se sumaron los montos)")
resultado("SLV-F1-006", n["filas"], n["f006"], "casos año × sec_ejec")
resultado("SLV-F1-007", n["filas"], n["f007"], "devoluciones y extornos; se conservan")

COLUMNAS_SILVER = ["anio", "mes", "anio_parcial", "sec_ejec", "ejecutora", "ejecutora_nombre", "tipo_municipalidad", "ubigeo",
                   *[c for c in t.columns if c.startswith(("departamento_", "provincia_", "distrito_"))],
                   *[c for c in t.columns if c.startswith(("fuente_", "rubro", "tipo_recurso", "generica", "subgenerica", "especifica"))],
                   *MONTOS, "_alertas", "_filas_origen", "_run_id", "_fecha_proceso_utc", "_fuente",
                   "_archivo_origen", "_sha256_origen", "_fecha_descarga"]
COLUMNAS_CUARENTENA = [c for c in COLUMNAS_SILVER if c != "_alertas"] + ["_regla", "_motivo"]
SENSIBLES = re.compile(r"correo|e_?mail|telef|celular|fax|dni|documento|alcalde|apellido|direcc|domicilio|usuario|web", re.I)
sensibles = [c for c in set(COLUMNAS_SILVER + COLUMNAS_CUARENTENA) if SENSIBLES.search(c)]
resultado("SLV-TODAS-001", len(set(COLUMNAS_SILVER + COLUMNAS_CUARENTENA)), len(sensibles), "columnas revisadas")

# Cuadre (detalle en 5.1): filas por archivo y montos por año
dest = a_pandas(tabla.groupBy("_archivo_origen", "_destino").agg(F.sum("_filas_origen").alias("n")))
cuadre = (a_pandas(crudo.groupBy("_archivo_origen").count()).rename(columns={"count": "carga_cruda"})
          .merge(dest.pivot_table(index="_archivo_origen", columns="_destino", values="n", aggfunc="sum").reset_index(), how="left").fillna(0))
for c in ("silver", "cuarentena"):
    cuadre[c] = cuadre.get(c, 0)
anio_de = dict(zip(f1["ruta"], f1["anio"]))
cuadre["fuera_de_alcance"] = cuadre["_archivo_origen"].map(lambda r: int(fuera.loc[anio_de[r], "fuera_de_alcance"]))
cuadre["cuadra"] = cuadre["carga_cruda"] == cuadre["silver"] + cuadre["cuarentena"] + cuadre["fuera_de_alcance"]
m_silver = a_pandas(tabla.groupBy("anio").agg(*[F.sum(c).alias(f"silver_{c}") for c in MONTOS]).orderBy("anio")).set_index("anio")
cm = agg[[f"bronze_{c}" for c in MONTOS]].join(m_silver)
cm["cuadra"] = reduce(lambda a, b: a & b, [cm[f"bronze_{c}"] == cm[f"silver_{c}"] for c in MONTOS])
resultado("SLV-F1-008", len(cm), int((~cm["cuadra"]).sum()), "años cuyos montos no cuadran")
resultado("SLV-TODAS-002", int(cuadre["carga_cruda"].sum()), int((~cuadre["cuadra"]).sum()), "archivos que no cuadran")

display(pd.DataFrame(VERIF)[["regla", "accion", "filas_evaluadas", "filas_fallidas", "estado", "nota"]].set_index("regla"))
if sensibles:
    raise RuntimeError(f"SLV-TODAS-001: columnas sensibles en la salida: {sensibles}")
if not (cuadre["cuadra"].all() and cm["cuadra"].all()):
    raise RuntimeError("Cuadre (SLV-F1-008 / SLV-TODAS-002): no coincide; no se escribe nada (ver 5.1)")

,accion,filas_evaluadas,filas_fallidas,estado,nota
regla,,,,,
SLV-F1-001,cuarentena,12746440,0,OK,
SLV-F1-002,cuarentena,12746440,0,OK,
SLV-F1-003,cuarentena,12746440,0,OK,
SLV-F1-004,registrar,15204055,2457615,ALERTA,"E: 927,613 · R: 1,525,420 · mancomunidades: 4,582"
SLV-F1-005,registrar,12746440,74,ALERTA,37 filas consolidadas (se sumaron los montos)
SLV-F1-006,registrar,12746403,0,OK,casos año × sec_ejec
SLV-F1-007,registrar,12746403,72301,ALERTA,devoluciones y extornos; se conservan
SLV-TODAS-001,detener,43,0,OK,columnas revisadas
SLV-F1-008,detener,18,0,OK,años cuyos montos no cuadran


### 5.1 Cuadre

- **Filas, por archivo:** carga cruda = Silver (filas de origen, contando las consolidadas) + cuarentena + fuera de alcance.
- **Montos, por año:** la suma de PIA, PIM y recaudado de las municipalidades en Bronze = la suma en la tabla limpia, en millones de soles.

In [19]:
cuadre["archivo"] = cuadre["_archivo_origen"].str.split("/").str[-1]
display(cuadre[["archivo", "carga_cruda", "silver", "cuarentena", "fuera_de_alcance", "cuadra"]]
        .astype({c: int for c in ["carga_cruda", "silver", "cuarentena", "fuera_de_alcance"]}).set_index("archivo"))
display(pd.concat([(cm[[x for c in MONTOS for x in (f"bronze_{c}", f"silver_{c}")]].astype(float) / 1e6).round(1), cm["cuadra"]], axis=1))

,carga_cruda,silver,cuarentena,fuera_de_alcance,cuadra
archivo,,,,,
2009-Ingreso.csv,1235428,1080470,0,154958,True
2010-Ingreso.csv,1320104,1159789,0,160315,True
2011-Ingreso.csv,1348362,1185119,0,163243,True
2012-Ingreso.csv,759578,638761,0,120817,True
2013-Ingreso.csv,721663,590028,0,131635,True
2014-Ingreso.csv,753998,611366,0,142632,True
2015-Ingreso.csv,781007,632282,0,148725,True
2016-Ingreso.csv,764731,613187,0,151544,True
2017-Ingreso.csv,683816,550641,0,133175,True


,bronze_monto_pia,silver_monto_pia,bronze_monto_pim,silver_monto_pim,bronze_monto_recaudado,silver_monto_recaudado,cuadra
anio,,,,,,,
2009,11999.8,11999.8,20695.9,20695.9,18496.6,18496.6,True
2010,9138.8,9138.8,20936.5,20936.5,20161.7,20161.7,True
2011,10793.5,10793.5,21314.0,21314.0,20858.4,20858.4,True
2012,14826.2,14826.2,27204.6,27204.6,26275.6,26275.6,True
2013,17559.5,17559.5,28949.6,28949.6,26565.0,26565.0,True
2014,16353.0,16353.0,27065.6,27065.6,25855.0,25855.0,True
2015,15230.7,15230.7,23981.7,23981.7,23379.8,23379.8,True
2016,14269.9,14269.9,26444.9,26444.9,25955.9,25955.9,True
2017,14553.4,14553.4,29055.1,29055.1,28792.3,28792.3,True


## 6. Escritura en Parquet

- **Silver:** `data/silver/siaf_ingreso/`, particionado por `anio`. Se sobrescribe en cada ejecución.
- **Cuarentena:** `data/quarantine/siaf_ingreso/` (se escribe aunque esté vacía, para que exista el esquema).
- **Trazabilidad:** `_run_id`, `_fecha_proceso_utc`, `_fuente`, el archivo de origen con su `sha256` y fecha de descarga, y `_filas_origen`.

In [20]:
fecha_proceso = datetime.now(timezone.utc)
final = (tabla.withColumn("_run_id", F.lit(RUN_ID)).withColumn("_fecha_proceso_utc", F.lit(fecha_proceso))
              .withColumn("_fuente", F.lit("F1")))
DESTINO, DESTINO_Q = ruta("silver") / TABLA, ruta("quarantine") / TABLA
final.where(F.col("_destino") == "silver").select(*COLUMNAS_SILVER).write.mode("overwrite").partitionBy("anio").parquet(str(DESTINO))
final.where(F.col("_destino") == "cuarentena").select(*COLUMNAS_CUARENTENA).write.mode("overwrite").parquet(str(DESTINO_Q))

leido = spark.read.parquet(str(DESTINO))
n_leido, n_q = leido.count(), spark.read.parquet(str(DESTINO_Q)).count()
assert n_leido == n["silver"] and n_q == n["cuarentena"], "Lo escrito no coincide con lo esperado"
print(f"Silver: {n_leido:,} filas en {DESTINO.relative_to(RAIZ)} · Cuarentena: {n_q:,} · Fuera de alcance: {int(fuera['fuera_de_alcance'].sum()):,}")
print("Columnas:", len(leido.columns))

Silver: 12,746,403 filas en data/silver/siaf_ingreso · Cuarentena: 0 · Fuera de alcance: 2,457,615
Columnas: 41


## 7. Bitácora y auditoría

Una línea en `data/silver/bitacora_ejecuciones.jsonl` con el bloque `volumenes` (mismo formato que Bronze), el resultado de cada regla y los cambios de la limpieza. Luego se reconstruye `data/auditoria/volumenes.parquet`.

In [21]:
fin = datetime.now(timezone.utc)
n_arch, n_bytes = auditoria.tamano_en_disco(DESTINO)
vol = auditoria.volumen("silver", "F1", TABLA, filas=n_leido, archivos=n_arch, bytes=n_bytes, fecha_datos=f1["fecha_descarga"].max(),
                        filas_carga_cruda=int(agg["filas"].sum()), archivos_origen=len(f1),
                        filas_fuera_de_alcance=int(fuera["fuera_de_alcance"].sum()), filas_cuarentena=n_q,
                        filas_consolidadas=int(n["f005"]))
cambios = pd.DataFrame(CAMBIOS).groupby("paso", sort=False)["valores_cambiados"].sum() if CAMBIOS else pd.Series(dtype=int)
entrada = {"run_id": RUN_ID, "notebook": "03_siaf", "capa": "silver", "inicio_utc": inicio.isoformat(), "fin_utc": fin.isoformat(),
           "tabla": TABLA, "anios": sorted(f1["anio"].tolist()), "volumenes": [vol],
           "destino": str(DESTINO.relative_to(RAIZ)), "destino_cuarentena": str(DESTINO_Q.relative_to(RAIZ)),
           "cambios_limpieza": {k: int(v) for k, v in cambios.items()}, "verificaciones": VERIF}
auditoria.escribir_bitacora("silver", entrada)
print("Bitácora actualizada:", RUN_ID, f"({(fin - inicio).total_seconds() / 60:.1f} min)")
df_aud = auditoria.construir_auditoria()
display(auditoria.ultimo_volumen(df_aud)[["capa", "fuente", "tabla", "archivos", "filas", "mb", "fecha_datos", "fin_utc"]])

Bitácora actualizada: silver_siaf_20261122T203648Z_543d24 (5.5 min)


,capa,fuente,tabla,archivos,filas,mb,fecha_datos,fin_utc
0,bronze,F1,siaf_ingreso,18,15204055,7336.2,2026-11-22,2026-11-22 17:37:44.000000+00:00
1,silver,F1,siaf_ingreso,432,12746403,200.8,2026-11-22,2026-11-22 20:42:19.804868+00:00
2,bronze,F2,sismepre_predial,7,493843,59.1,2026-11-22,2026-11-22 17:37:44.000000+00:00
3,bronze,F3,renamu,1109,2240471,354.5,2026-11-20,2026-11-22 17:37:44.000000+00:00
4,silver,F3,renamu_finanzas,43,29841,2.3,2026-11-20,2026-11-22 19:48:42.412889+00:00


## 8. Resumen final

**Tabla `siaf_ingreso`:** 12 746 403 filas y 41 columnas, en `data/silver/siaf_ingreso/`, particionada por `anio` (2009–2026). Ocupa 201 MB, frente a los 7,3 GB de los CSV.

- **Recorrido de las filas:** 15 204 055 filas de 18 archivos → **2 457 615 fuera de alcance** (E: 927 613, R: 1 525 420, mancomunidades: 4 582) → 12 746 440 de municipalidades → 0 en cuarentena → 37 filas consolidadas (74 de origen) → **12 746 403 en Silver**.
- **Cuadre:** cada archivo cuadra (carga = Silver + cuarentena + fuera de alcance), y la suma de PIA, PIM y recaudado de las municipalidades es la misma en Bronze y en Silver los 18 años.
- **Limpieza:**
  - se quitan 6 columnas sin información (nivel de gobierno, sector y pliego);
  - se quitan los espacios sobrantes, sobre todo en los nombres;
  - se arma el `ubigeo`;
  - se agrega `tipo_municipalidad` (Lima siempre PROVINCIAL);
  - los montos pasan a `decimal(18,2)`;
  - se consolidan las copias que dejó el cambio de nombre de una ejecutora a mitad de año;
  - se marca `anio_parcial` (2026 llega hasta noviembre).
- **Alertas** (las filas entran a Silver, con el id de la regla en `_alertas`):
  - `SLV-F1-005`, 37 filas consolidadas;
  - `SLV-F1-007`, 72 301 filas con recaudado negativo (devoluciones y extornos).

  `SLV-F1-004` solo cuenta lo que quedó fuera de alcance.
- **Se conserva tal como viene:**
  - los nombres de cada año: 11 casos de cambio de nombre dentro del año, que en Gold se resuelven con el último nombre;
  - las filas con todos los montos en 0;
  - el grano mensual: los totales anuales del PIA y el PIM se arman en Gold.
- **Columnas:**
  - período: `anio`, `mes`, `anio_parcial`;
  - ejecutora: `sec_ejec`, `ejecutora`, `ejecutora_nombre`, `tipo_municipalidad`;
  - ubicación: `ubigeo` y los códigos y nombres de departamento, provincia y distrito;
  - clasificador: fuente, rubro, tipo de recurso, genérica, subgenérica, específica y sus detalles, con código y nombre;
  - montos: `monto_pia`, `monto_pim`, `monto_recaudado`;
  - control: `_alertas`, `_filas_origen`.
- **Trazabilidad:** `_run_id`, `_fecha_proceso_utc`, `_fuente` y, por fila, `_archivo_origen`, `_sha256_origen` y `_fecha_descarga`.
- **Auditoría:** Silver F1 aparece en `data/auditoria/volumenes.parquet` debajo de Bronze F1 (15,2 → 12,7 millones de filas; 7,3 GB → 201 MB).

In [22]:
muni.unpersist(); rep.unpersist(); tabla.unpersist()
spark.stop()